# Prithvi-EO Fine-Tuning & Quantization Spike
Run this notebook in Google Colab (with a T4 GPU) to test the end-to-end pipeline. It will automatically detect the correct model name from the registry, run a dummy training loop, and export a quantized model.

In [8]:
!pip install -q torch torchvision onnx onnxruntime onnxruntime-tools terratorch transformers


Found existing installation: numpy 2.5.2
Uninstalling numpy-2.5.2:
  Successfully uninstalled numpy-2.5.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 193.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for numpy: filename=numpy-1.26.4-cp313-cp313-linux_x86_64.whl size=6917446 sha256=18f3e094d362aa7f451e120cc1a75f87c8efd5b30321ba202368dace7ddfd10f
  Stored in directory: /tmp/pip-ephem-wheel-cache-w9zmu9w6/wheels/8b/2d/9f/b6b46373f328e2ef50388915d351ccacbedac929459b5459bf
Successfully built numpy
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
terratorch 1.2.13 requires numpy>=2.2, but you have numpy 1.26.4 which is incompatible.
rioxarray 0.23.0 requires numpy>=2, but you have numpy 1.2

In [10]:
import os
import json
import torch
from huggingface_hub import hf_hub_download

# You might need these small dependencies if you haven't installed them yet
!pip install -q timm einops

print("Downloading Prithvi 2.0 (300M) custom architecture and weights...")
hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-EO-2.0-300M", filename="prithvi_mae.py", local_dir=".")
hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-EO-2.0-300M", filename="config.json", local_dir=".")
weights_path = hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-EO-2.0-300M", filename="Prithvi_EO_V2_300M.pt", local_dir=".")

# Import the architecture directly from the downloaded file
from prithvi_mae import PrithviViT
import torch.nn as nn

# Load their config (extracting the nested pretrained_cfg dictionary)
with open("config.json", "r") as f:
    config = json.load(f)["pretrained_cfg"]

# Initialize the raw Backbone exactly how they define it
backbone = PrithviViT(
    img_size=config["img_size"],
    patch_size=config["patch_size"],
    num_frames=config["num_frames"],
    in_chans=config["in_chans"],
    embed_dim=config["embed_dim"],
    depth=config["depth"],
    num_heads=config["num_heads"],
    mlp_ratio=config["mlp_ratio"],
)



# Load the weights (stripping out the MAE pre-training decoder stuff)
state_dict = torch.load(weights_path, map_location="cpu")
if "model" in state_dict:
    state_dict = state_dict["model"]
state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items() if "decoder" not in k and "mask_token" not in k}
backbone.load_state_dict(state_dict, strict=False)

# Wrap it in our Segmentation Head Spike
class PrithviSegmentationSpike(nn.Module):
    def __init__(self, backbone, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Conv2d(self.backbone.embed_dim, num_classes, kernel_size=1)

    def forward(self, x):
        feats = self.backbone(x)

        # Reshape the sequence back into a 2D feature map
        b, seq, dim = feats.shape
        h = w = int(seq ** 0.5)
        feats = feats.transpose(1, 2).reshape(b, dim, h, w)

        # Upsample back to the original image resolution (224x224)
        feats = torch.nn.functional.interpolate(feats, size=(224, 224), mode='bilinear', align_corners=False)
        return self.head(feats)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = PrithviSegmentationSpike(backbone).to(device)

print("Successfully loaded Prithvi-EO-2.0-300M custom architecture and moved to device!")


Successfully loaded Prithvi-EO-2.0-300M custom architecture and moved to device!


In [21]:
# 3. Setup a Custom Dummy Dataset for the Spike
from torch.utils.data import DataLoader, Dataset

class CustomSatelliteDataset(Dataset):
    def __len__(self):
        return 16 # Small dataset for spike

    def __getitem__(self, idx):
        # Prithvi expects [Bands, Height, Width] (it auto-adds Time)
        return {
            'image': torch.randn(6, 1, 224, 224), # Time=1 is now in the 5D correct spot!
            'mask': torch.randint(0, 2, (224, 224))
        }

dataset = CustomSatelliteDataset()
dataloader = DataLoader(dataset, batch_size=4)


In [22]:
# 4. Training Loop Spike
device = 'cuda' if torch.cuda.is_available() else 'cpu'
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("Starting training loop...")
model.train()
for epoch in range(2): # Just 2 epochs for the spike
    for i, batch in enumerate(dataloader):
        inputs, targets = batch['image'].to(device), batch['mask'].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        # Dummy loss for spike
        loss = outputs.sum()
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}, Batch {i+1} completed. Dummy loss: {loss.item():.4f}")

print("Training complete! Saving weights...")
torch.save(model.state_dict(), "prithvi_spike_finetuned.pt")

Starting training loop...


AttributeError: 'tuple' object has no attribute 'shape'

In [13]:
# Phase 4: Benchmark & Validate
import time, psutil, onnxruntime as ort
import numpy as np

# 1. Update the filename to match the output of the INT8 quantization cell
session = ort.InferenceSession("prithvi_int8.onnx", providers=["CPUExecutionProvider"])
session.set_providers(["CPUExecutionProvider"], [{"intra_op_num_threads": 2}])

# 2. Changed shape to [1, 6, 224, 224] to match
dummy_input = np.random.randn(1, 6, 224, 224).astype(np.float32)

mem_before = psutil.Process().memory_info().rss / (1024**2)
start = time.time()
for _ in range(20):
    outputs = session.run(None, {"input_image": dummy_input})
elapsed = (time.time() - start) / 20
mem_after = psutil.Process().memory_info().rss / (1024**2)

print(f"Avg latency: {elapsed:.2f}s | Peak RSS delta: {mem_after - mem_before:.0f}MB")


NoSuchFile: [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from prithvi_300m_int8.onnx failed:Load model prithvi_300m_int8.onnx failed. File doesn't exist

In [ ]:
# 5. ONNX Export
print("Exporting to ONNX...")
model.eval()

# Changed shape to [1, 6, 224, 224] and added .to(device)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dummy_input = torch.randn(1, 6, 224, 224).to(device)

onnx_path = "prithvi_fp32.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input_image'],
    output_names=['segmentation_mask']
)
print(f"Successfully exported to {onnx_path}")


In [ ]:
# 6. INT8 Quantization
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

model_quant = "prithvi_int8.onnx"
print("Running dynamic INT8 quantization...")

quantize_dynamic(
    model_input=onnx_path,
    model_output=model_quant,
    weight_type=QuantType.QUInt8,
    optimize_model=True
)

print(f"Quantized model saved to {model_quant}")

size_fp32 = os.path.getsize(onnx_path) / (1024 * 1024)
size_int8 = os.path.getsize(model_quant) / (1024 * 1024)
print(f"\n--- Compression Results ---")
print(f"Original FP32 Size: {size_fp32:.2f} MB")
print(f"Quantized INT8 Size: {size_int8:.2f} MB")